In [9]:
import transformers
from openai import OpenAI
from dotenv import load_dotenv
import json

In [16]:
load_dotenv(override=True)
#client = OpenAI()  # uses OPENAI_API_KEY from env



True

In [57]:
tools = [{
        "type": "function",
        "function": {
            "name": "apply_code_fix",
            "description": "Applies a specific fix to a file to resolve a SonarQube issue.",
            "parameters": {
                "type": "object",
                "properties": {
                    "file_path": {"type": "string"},
                    "new_content": {"type": "string", "description": "The complete updated content of the file."}
                },
                "required": ["file_path", "fixed_code"]
            }
        }
    }]


In [72]:
response = client.chat.completions.create(model="gpt-4o", messages=messages, tools=tools)
print(response.choices[0].message.content)

None


In [ ]:
done = False
while not done:
    response = client.chat.completions.create(model="gpt-4o", messages=messages, tools=tools)
    print(response.choices[0].mess)
    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        arguments = json.loads(message.tool_calls[0].function.arguments)
        print(arguments)
        results = apply_fix_to_disk(**arguments)
        messages.append(message)
        messages.append(results)
        
    else:
        done = True
response.choices[0].message.content

!!!!!!!!!!!!!!!!!!!!!!
Final LLM Response: ok


In [87]:
print(messages)

[{'role': 'system', 'content': 'You are a senior developer. Use the apply_code_fix tool to submit your changes.'}, {'role': 'user', 'content': '\n"Change this default value to "None" and initialize this parameter inside the function/method.",\n"component": "mainn.py"\n"reply ok if finished"\n"code ": "import os\nimport sys  # Σφάλμα 1: Unused import (Code Smell)\n\ndef authenticate_user(username, password):\n    # Σφάλμα 2: Hardcoded credentials (Security Hotspot)\n    admin_pass = "super_secret_password_123!" \n\n    if username == "admin":\n        if password == admin_pass:\n            print("Admin logged in")\n            return True\n        else:\n            print("Wrong password")\n            return False\n    else:\n        return False\n\n# Σφάλμα 3: Mutable default argument (Bug - Πολύ επικίνδυνο στην Python)\ndef add_item_to_cart(item, cart=[]):\n    cart.append(item)\n    print(f"Added {item} to cart.")\n    return cart\n\ndef calculate_discount(price, customer_type):\n 

In [94]:
system_message = "You are a senior developer. Use the apply_code_fix tool to submit your changes."
user_message = """
"Change this default value to \"None\" and initialize this parameter inside the function/method.",
"component": "mainn.py"
"reply ok if finished"
"code ": "import os
import sys  # Σφάλμα 1: Unused import (Code Smell)

def authenticate_user(username, password):
    # Σφάλμα 2: Hardcoded credentials (Security Hotspot)
    admin_pass = "super_secret_password_123!" 
    
    if username == "admin":
        if password == admin_pass:
            print("Admin logged in")
            return True
        else:
            print("Wrong password")
            return False
    else:
        return False

# Σφάλμα 3: Mutable default argument (Bug - Πολύ επικίνδυνο στην Python)
def add_item_to_cart(item, cart=[]):
    cart.append(item)
    print(f"Added {item} to cart.")
    return cart

def calculate_discount(price, customer_type):
    # Σφάλμα 4: High Cognitive Complexity (Code Smell - Πολλά if/else)
    discount = 0
    if price > 100:
        if customer_type == "VIP":
            discount = 0.20
        elif customer_type == "Regular":
            discount = 0.10
        else:
            if price > 500:
                discount = 0.05
    else:
        if customer_type == "VIP":
            discount = 0.05
            
    final_price = price - (price * discount)
    return final_price

def main():
    add_item_to_cart("Laptop")
    add_item_to_cart("Mouse")
    print(calculate_discount(150, "VIP"))

if __name__ == "__main__":
    main()"
"""
messages = [{"role": "system", "content": system_message},{"role": "user", "content": user_message}]
messages

[{'role': 'system',
  'content': 'You are a senior developer. Use the apply_code_fix tool to submit your changes.'},
 {'role': 'user',
  'content': '\n"Change this default value to "None" and initialize this parameter inside the function/method.",\n"component": "mainn.py"\n"reply ok if finished"\n"code ": "import os\nimport sys  # Σφάλμα 1: Unused import (Code Smell)\n\ndef authenticate_user(username, password):\n    # Σφάλμα 2: Hardcoded credentials (Security Hotspot)\n    admin_pass = "super_secret_password_123!" \n\n    if username == "admin":\n        if password == admin_pass:\n            print("Admin logged in")\n            return True\n        else:\n            print("Wrong password")\n            return False\n    else:\n        return False\n\n# Σφάλμα 3: Mutable default argument (Bug - Πολύ επικίνδυνο στην Python)\ndef add_item_to_cart(item, cart=[]):\n    cart.append(item)\n    print(f"Added {item} to cart.")\n    return cart\n\ndef calculate_discount(price, customer_type

In [76]:
import os

def apply_fix_to_disk(file_path: str, new_content: str):
    """
    Physically overwrites the file with the fixed version.
    """
    try:
        # Ensure the directory exists
        #os.makedirs(os.path.dirname(file_path), exist_ok=True)
        
        with open(file_path, "w+", encoding="utf-8") as f:
            f.write(new_content)
        return "changed success"
    except Exception as e:
        print(f"Error writing to file {file_path}: {e}")
        return "changed failed"

In [ ]:
from agents.fixer import FixerAgent
agent = FixerAgent(model_name="gpt-5.1-nano",url = None, token=None)

In [20]:
import os

client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

# This will show you exactly what strings to use in your FixerAgent
models = client.models.list()
for model in models:
    print(f"Available Model: {model.id}")

Available Model: models/gemini-2.5-flash
Available Model: models/gemini-2.5-pro
Available Model: models/gemini-2.0-flash
Available Model: models/gemini-2.0-flash-001
Available Model: models/gemini-2.0-flash-lite-001
Available Model: models/gemini-2.0-flash-lite
Available Model: models/gemini-2.5-flash-preview-tts
Available Model: models/gemini-2.5-pro-preview-tts
Available Model: models/gemma-3-1b-it
Available Model: models/gemma-3-4b-it
Available Model: models/gemma-3-12b-it
Available Model: models/gemma-3-27b-it
Available Model: models/gemma-3n-e4b-it
Available Model: models/gemma-3n-e2b-it
Available Model: models/gemma-4-26b-a4b-it
Available Model: models/gemma-4-31b-it
Available Model: models/gemini-flash-latest
Available Model: models/gemini-flash-lite-latest
Available Model: models/gemini-pro-latest
Available Model: models/gemini-2.5-flash-lite
Available Model: models/gemini-2.5-flash-image
Available Model: models/gemini-3-pro-preview
Available Model: models/gemini-3-flash-previe